![image.png](https://i.imgur.com/a3uAqnb.png)

# BART Fine-tuning for Text Summarization
In this homework, you will implement **BART fine-tuning for text summarization** using the BBC News dataset. This project will help you understand transformer-based sequence-to-sequence models, fine-tuning techniques, and evaluation metrics for summarization tasks.

## 📌 Project Overview
- **Task**: Fine-tune BART model for news article summarization
- **Model**: Facebook's BART-base (Bidirectional and Auto-Regressive Transformers)
- **Dataset**: BBC News Summary Dataset
- **Goal**: Generate concise, coherent summaries of news articles

## 📚 Learning Objectives
By completing this assignment, you will:
- Understand transformer-based sequence-to-sequence architectures
- Learn how to fine-tune pre-trained language models
- Practice data preprocessing for summarization tasks
- Implement training loops with Hugging Face Transformers
- Evaluate summarization quality using ROUGE metrics
- Compare pre-trained vs fine-tuned model performance

## 1️⃣ Initial Setup and Data Download

**Task**: Download the BBC News summarization dataset and explore its structure.

**Requirements**:
- Download dataset using KaggleHub
- Explore directory structure
- Understand the organization of articles and summaries

In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter
import re

# TODO: Download the BBC News Summary dataset
path = kagglehub.dataset_download("pariza/bbc-news-summary")

# TODO: Create function to explore dataset structure
def explore_dataset_structure(base_path):
    base_path = Path(base_path)
    articles_dir = base_path / "BBC News Summary" / "News Articles"
    summaries_dir = base_path / "BBC News Summary" / "Summaries"

    # TODO: Get all article categories
    article_categories = [d.name for d in articles_dir.iterdir() if d.is_dir()]

    # TODO: Count total articles across all categories
    total_articles = 0
    for category in article_categories:
        article_files = list((articles_dir / category).glob("*.txt"))
        total_articles += len(article_files)

    print(f"Total articles loaded: {total_articles}")
    return articles_dir, summaries_dir, article_categories

# TODO: Explore the dataset structure
articles_dir, summaries_dir, categories = explore_dataset_structure(path)

# TODO: Load and display sample article and summary
sample_category = categories[0]
article_files = list((articles_dir / sample_category).glob("*.txt"))
summary_files = list((summaries_dir / sample_category).glob("*.txt"))

# TODO: Read sample article
with open(article_files[0], 'r', encoding='utf-8', errors='ignore') as f:
    sample_article = f.read()

# TODO: Read corresponding summary
sample_summary_file = summaries_dir / sample_category / article_files[0].name
with open(sample_summary_file, 'r', encoding='utf-8', errors='ignore') as f:
    sample_summary = f.read()

# TODO: Display basic statistics
print(f"Sample article length: {len(sample_article.split())} words")
print(f"Sample summary length: {len(sample_summary.split())} words")

## 2️⃣ Data Loading and Preprocessing

**Task**: Create a comprehensive dataset class to load and clean all articles and summaries.

**Requirements**:
- Implement text cleaning function
- Load all articles and summaries from all categories
- Create paired article-summary dataset
- Handle encoding issues and missing files
- Display dataset statistics and category distribution

In [ ]:
# TODO: Install required libraries (uncomment if needed)
#!pip install transformers datasets evaluate rouge-score

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from datasets import Dataset
import pandas as pd
from pathlib import Path
import re
from tqdm import tqdm
import numpy as np

In [ ]:
# TODO: Create comprehensive dataset class for BBC News
class BBCNewsDataset:
    def __init__(self, base_path):
        self.base_path = Path(base_path)
        self.articles_dir = self.base_path / "BBC News Summary" / "News Articles"
        self.summaries_dir = self.base_path / "BBC News Summary" / "Summaries"
        self.data = self.load_all_data()

    def clean_text(self, text):
        """TODO: Implement basic text cleaning"""
        # TODO: Remove extra whitespace
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()
        return text

    def load_all_data(self):
        """TODO: Load all articles and summaries from all categories"""
        data = []
        categories = ['politics', 'entertainment', 'sport', 'tech', 'business']

        for category in categories:
            article_path = self.articles_dir / category
            summary_path = self.summaries_dir / category

            # TODO: Get all article files in category
            article_files = list(article_path.glob("*.txt"))

            for article_file in article_files:
                try:
                    # TODO: Read article text
                    with open(article_file, 'r', encoding='utf-8', errors='ignore') as f:
                        article_text = f.read()

                    # TODO: Read corresponding summary
                    summary_file = summary_path / article_file.name
                    with open(summary_file, 'r', encoding='utf-8', errors='ignore') as f:
                        summary_text = f.read()

                    # TODO: Clean both texts
                    article_clean = self.clean_text(article_text)
                    summary_clean = self.clean_text(summary_text)

                    # TODO: Only include valid pairs
                    if len(article_clean.strip()) > 0 and len(summary_clean.strip()) > 0:
                        data.append({
                            'category': category,
                            'filename': article_file.name,
                            'article': article_clean,
                            'summary': summary_clean
                        })
                except Exception as e:
                    print(f"Error processing {article_file}: {e}")
                    continue

        print(f"Loaded {len(data)} article-summary pairs")
        return data

# TODO: Load the dataset
bbc_dataset = BBCNewsDataset(path)

# TODO: Convert to pandas DataFrame for easier handling
df = pd.DataFrame(bbc_dataset.data)
print(f"Dataset shape: {df.shape}")
print(f"Categories: {df['category'].value_counts()}")

## 3️⃣ Model Initialization and Setup

**Task**: Initialize the BART model and tokenizer for sequence-to-sequence learning.

**Requirements**:
- Load pre-trained BART-base model and tokenizer
- Set up device configuration (GPU/CPU)
- Display model information and parameter count
- Prepare model for fine-tuning

In [ ]:
# TODO: Initialize BART model and tokenizer
model_name = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# TODO: Check if GPU is available and move model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Model: {model_name}")
print(f"Device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4️⃣ Data Preprocessing and Tokenization

**Task**: Prepare the dataset for BART training by tokenizing inputs and targets.

**Requirements**:
- Implement preprocessing function for sequence-to-sequence data
- Tokenize articles (inputs) and summaries (targets)
- Set appropriate maximum lengths for input and target sequences
- Create train-validation split
- Apply preprocessing to both splits

In [ ]:
# TODO: Define preprocessing function for sequence-to-sequence data
def preprocess_function(examples, max_input_length=1024, max_target_length=128):
    """TODO: Tokenize inputs and targets for BART training"""
    # TODO: Tokenize inputs (articles)
    inputs = [article for article in examples["article"]]
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding=False
    )

    # TODO: Tokenize targets (summaries)
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=max_target_length,
        truncation=True,
        padding=False
    )

    # TODO: Add labels to model inputs
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# TODO: Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# TODO: Split dataset into train and validation
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

# TODO: Apply preprocessing to both datasets
train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

eval_dataset = eval_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=eval_dataset.column_names
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Eval dataset size: {len(eval_dataset)}")

## 5️⃣ Text Generation Function

**Task**: Implement a function to generate summaries using the BART model.

**Requirements**:
- Create generation function with proper parameters
- Handle tokenization and device placement
- Implement beam search for better quality
- Add length penalties and early stopping
- Return decoded summary text

In [ ]:
def generate_summary(text, max_length=128, min_length=30):
    """TODO: Generate summary using BART model"""
    # TODO: Tokenize input text
    inputs = tokenizer(
        text,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # TODO: Generate summary with beam search
    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_length,
            min_length=min_length,
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True
        )

    # TODO: Decode summary to text
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

## 6️⃣ Pre-training Evaluation

**Task**: Test the pre-trained BART model before fine-tuning to establish baseline performance.

**Requirements**:
- Generate summaries for sample articles using pre-trained model
- Compare generated summaries with ground truth
- Display examples from different categories
- Analyze baseline quality and identify areas for improvement

In [ ]:
# TODO: Test on a few examples from the dataset before training
print("Testing on sample articles:")
print("=" * 80)

for i in range(3):
    sample = df.iloc[i]
    article = sample['article']
    ground_truth = sample['summary']
    generated = generate_summary(article)

    print(f"\nSample {i+1} - Category: {sample['category']}")
    print(f"Article: {article}...")
    print(f"Ground Truth: {ground_truth}")
    print(f"Generated: {generated}")
    print("-" * 80)

## 7️⃣ Training Configuration

**Task**: Set up training arguments and data collator for BART fine-tuning.

**Requirements**:
- Configure training arguments with appropriate hyperparameters
- Set up data collator for sequence-to-sequence training
- Configure learning rate, batch size, and number of epochs
- Enable mixed precision training if GPU available
- Set up logging and evaluation strategies

In [ ]:
# TODO: Create data collator for sequence-to-sequence training
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

# TODO: Configure training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-bbc-news-summarization",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),  # TODO: Use mixed precision if GPU available
    logging_dir="./logs",
    logging_steps=50,
    save_strategy="epoch",
    report_to=[],
    eval_strategy="epoch",
    dataloader_num_workers=2,
    gradient_accumulation_steps=2,  # TODO: Effective batch size = 4 * 2 = 8
)

## 8️⃣ Evaluation Metrics Setup

**Task**: Configure ROUGE metrics for evaluating summarization quality.

**Requirements**:
- Load ROUGE evaluation metric
- Implement compute_metrics function for training
- Handle prediction decoding and label processing
- Calculate ROUGE-1, ROUGE-2, and ROUGE-L scores
- Format results as percentages

In [ ]:
import evaluate

# TODO: Load ROUGE metric for summarization evaluation
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    """TODO: Compute ROUGE metrics for summarization evaluation"""
    predictions, labels = eval_pred

    # TODO: Decode predictions to text
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # TODO: Replace -100 in the labels (can't decode padding tokens)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # TODO: Compute ROUGE scores
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    # TODO: Convert to percentages
    result = {key: value * 100 for key, value in result.items()}

    return {k: round(v, 4) for k, v in result.items()}

## 9️⃣ Trainer Initialization and Baseline Evaluation

**Task**: Initialize the Seq2SeqTrainer and evaluate pre-trained model performance.

**Requirements**:
- Create Seq2SeqTrainer with all components
- Evaluate baseline performance before fine-tuning
- Display baseline ROUGE scores
- Prepare for training process

In [ ]:
# TODO: Initialize trainer with all components
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
# TODO: Evaluate the pretrained model first to get baseline scores
baseline_results = trainer.evaluate()

print("Baseline ROUGE scores (before fine-tuning):")
for key, value in baseline_results.items():
    if 'rouge' in key.lower():
        print(f"{key}: {value:.4f}")

## 🔟 Model Training

**Task**: Fine-tune the BART model on the BBC News dataset.

**Requirements**:
- Start the training process using the configured trainer
- Monitor training progress and loss
- Track evaluation metrics during training
- Save model checkpoints automatically
- Handle potential memory issues with appropriate batch sizes

In [ ]:
# TODO: Start training the model
trainer.train()

## 1️⃣1️⃣ Model Saving and Loading

**Task**: Save the fine-tuned model and prepare for evaluation.

**Requirements**:
- Save the final trained model and tokenizer
- Load the fine-tuned model for evaluation
- Ensure model is moved to correct device
- Prepare for post-training evaluation

In [ ]:
# TODO: Save the final model and tokenizer
trainer.save_model("./bart-bbc-final")
tokenizer.save_pretrained("./bart-bbc-final")

In [ ]:
# TODO: Load the trained model for evaluation
model = AutoModelForSeq2SeqLM.from_pretrained("./bart-bbc-final")
tokenizer = AutoTokenizer.from_pretrained("./bart-bbc-final")
model = model.to(device)

## 1️⃣2️⃣ Post-Training Evaluation

**Task**: Evaluate the fine-tuned model and compare with baseline performance.

**Requirements**:
- Generate summaries using the fine-tuned model
- Compare with ground truth summaries
- Show examples from different categories
- Test on the same samples used for baseline evaluation

In [ ]:
# TODO: Test on the same examples after training
print("Testing on sample articles - After Training:")
print("=" * 80)

for i in range(3):
    sample = df.iloc[i]
    article = sample['article']
    ground_truth = sample['summary']
    generated = generate_summary(article)

    print(f"\nSample {i+1} - Category: {sample['category']}")
    print(f"Article: {article[:200]}...")
    print(f"Ground Truth: {ground_truth}")
    print(f"Generated: {generated}")
    print("-" * 80)

## 1️⃣3️⃣ Custom Article Testing

**Task**: Test the fine-tuned model on custom news articles to verify generalization.

**Requirements**:
- Create custom news article for testing
- Generate summary using fine-tuned model
- Analyze quality of generated summary
- Verify model's ability to generalize to new content
- Compare summary characteristics with training data

In [ ]:
# TODO: Test with custom article
custom_article = """
The government announced new policies to tackle climate change today.
The prime minister said that renewable energy investments will increase by 50% next year.
Solar and wind power projects will receive additional funding.
Environmental groups welcomed the announcement but said more action is needed.
The new policies include tax incentives for electric vehicle purchases and stricter emissions standards for industries.
Coal power plants will be phased out over the next decade, while nuclear energy capacity will be expanded.
Critics argue that the timeline is too slow to meet international climate commitments.
"""

print("Custom Article:")
print(custom_article.strip())
print("\nGenerated Summary:")
custom_summary = generate_summary(custom_article)
print(custom_summary)

## 📊 Evaluation Criteria

Your homework will be evaluated based on:

1. **Implementation Correctness (40%)**
   - Proper dataset loading and preprocessing
   - Correct BART model initialization and configuration
   - Working tokenization and data preparation pipeline
   - Successful model training and fine-tuning

2. **Model Performance (30%)**
   - Improvement in ROUGE scores after fine-tuning
   - Quality of generated summaries
   - Proper convergence during training
   - Meaningful comparison between pre-trained and fine-tuned models

3. **Code Quality and Analysis (30%)**
   - Clean, readable code with proper structure
   - Comprehensive evaluation and comparison
   - Good use of Hugging Face Transformers library
   - Proper handling of sequence-to-sequence training